# Notebook 2: Online Feature Store Setup

Sets up the Postgres-backed Online Service, registers stream feature views with continuous aggregation for all 4 entity velocity groups, and benchmarks freshness and lookup latency.

---

### Architecture

```
FRAUD_TRANSACTIONS (source of truth)
      │
      └──► REST Ingest API ──► Online Feature Store (Postgres)
                                  ├── Stream FVs: CONTINUOUS aggregation (< 2s)
                                  └── Batch FV: Customer profile (daily)
```

### Feature Coverage

| Entity | Stream aggregations | Notes |
|---|---|---|
| Customer | count, sum, max, min, approx_count_distinct — 1h/6h/24h/48h/1wk | Primary card-testing signal |
| Merchant | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Merchant under attack |
| Wallet DPAN | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Compromised card |
| IP Address | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Bot farm |
| Customer Profile | Lifetime stats, account age | Batch online FV, daily refresh |

Derived ratio features (velocity ratios, concentration, deviation) are computed inline in the SPCS scoring container.

> **Preview note:** Requires `snowflake-ml-python >= 1.41` and `SNOWFLAKE_PAT` env var.

In [ ]:
import importlib.metadata
v = importlib.metadata.version('snowflake-ml-python')
major, minor = int(v.split('.')[0]), int(v.split('.')[1])
if major >= 1 and minor >= 41:
    print(f'snowflake-ml-python {v} — OK')
else:
    raise RuntimeError(f'Requires >= 1.41. Found: {v}\nRun: pip install --upgrade snowflake-ml-python')

In [ ]:
import time, os, numpy as np, requests, random, concurrent.futures
from datetime import datetime
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import (
    FeatureStore, FeatureView, Entity, CreationMode,
    OnlineConfig, OnlineStoreType, StreamSource, StreamConfig, Feature
)
from snowflake.ml.feature_store.spec.enums import FeatureAggregationMethod
from snowflake.ml.feature_store import online_service as ofs_utils
from snowflake.snowpark.types import (
    StructType, StructField, StringType, FloatType,
    TimestampType, TimestampTimeZone
)

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA FEATURES').collect()
session.sql('USE WAREHOUSE FRAUD_OFS_TRAIN_WH').collect()

fs = FeatureStore(
    session=session,
    database='FRAUD_DEMO_DEV',
    name='FEATURE_STORE',
    default_warehouse='FRAUD_OFS_TRAIN_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

PAT = os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    print('WARNING: SNOWFLAKE_PAT not set. Set: export SNOWFLAKE_PAT="<token>"')
else:
    print('PAT: set')
print('Feature Store: FRAUD_DEMO_DEV.FEATURE_STORE')

## 2.1 Create Online Service

Provisions a Postgres-backed serving layer co-located with your Snowflake account. Created once per Feature Store — takes 3-5 minutes on first run.

Exposes REST ingest and query endpoints (public, PrivateLink, or SPCS-internal). For production, configure PrivateLink to keep traffic off the public internet.

In [ ]:
print('Creating Online Service...\nFirst creation takes 3-5 minutes. Polling until RUNNING.\n')

try:
    fs.create_online_service('FRAUD_MLOPS', 'FRAUD_MLOPS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Online service already exists — continuing')
    else:
        raise

status = fs.get_online_service_status()
start = time.time()
while status.status != 'RUNNING':
    print(f'  [{time.time()-start:.0f}s] {status.status} — waiting 30s...')
    time.sleep(30)
    status = fs.get_online_service_status()

print(f'Online Service RUNNING ({time.time()-start:.0f}s)')

HEADERS = {'Authorization': f'Snowflake Token="{PAT}"', 'Content-Type': 'application/json'}
QUERY_URL  = ofs_utils.endpoint_url(status, 'query')
INGEST_URL = ofs_utils.endpoint_url(status, 'ingest')
print(f'Query URL:  {QUERY_URL}')
print(f'Ingest URL: {INGEST_URL}')

## 2.2 Register Entities

In [ ]:
customer_entity = Entity(name='FRAUD_CUSTOMER',    join_keys=['CUSTOMER_ID'])
merchant_entity = Entity(name='FRAUD_MERCHANT',    join_keys=['MERCHANT_ID'])
dpan_entity     = Entity(name='FRAUD_WALLET_DPAN', join_keys=['WALLET_DPAN'])
ip_entity       = Entity(name='FRAUD_IP_ADDRESS',  join_keys=['IP_ADDRESS'])

for entity in [customer_entity, merchant_entity, dpan_entity, ip_entity]:
    try:
        fs.register_entity(entity)
        print(f'  Registered: {entity.name}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'  Exists: {entity.name}')
        else:
            raise

## 2.3 Register Stream Source

Defines the event schema for real-time transaction ingestion. Every transaction is sent here via REST to trigger continuous velocity feature updates.

In [ ]:
txn_stream = StreamSource(
    name='FRAUD_TXN_EVENTS',
    schema=StructType([
        StructField('CUSTOMER_ID', StringType()),
        StructField('MERCHANT_ID', StringType()),
        StructField('WALLET_DPAN', StringType()),
        StructField('IP_ADDRESS',  StringType()),
        StructField('AMOUNT_USD',  FloatType()),
        StructField('IS_GBR',      FloatType()),
        StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
    ]),
    desc='Real-time fraud transaction events',
)

try:
    fs.register_stream_source(txn_stream)
    print('Stream source registered: FRAUD_TXN_EVENTS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Stream source already exists — continuing')
    else:
        raise

## 2.4 Create Stream Feature Views (CONTINUOUS Aggregation)

`FeatureAggregationMethod.CONTINUOUS` maintains running aggregates updated on every ingest event — no refresh cycle, no warehouse, < 2s end-to-end freshness.

**Why this matters:** A card-testing burst of 5 transactions spanning 30 seconds is invisible to a 33-39s refresh cycle. With CONTINUOUS aggregation, velocity counts update after each transaction — the model detects the burst from transaction 2 onwards.

> **approx_count_distinct note:** Distinct-value features (distinct merchants, DPANs) use HyperLogLog (~6.5% RSE at default precision). For fraud velocity signals this is negligible. Use `precision=14` to reduce RSE to ~0.8% if needed.

In [ ]:
online_cfg = OnlineConfig(enable=True, store_type=OnlineStoreType.POSTGRES)
stream_cfg = StreamConfig(stream_source=txn_stream)

# ── Customer velocity (primary card-testing signal) ──────────────────────────
customer_fv = FeatureView(
    name='FRAUD_CUSTOMER_VELOCITY',
    entities=[customer_entity],
    stream_config=stream_cfg,
    timestamp_col='EVENT_TS',
    refresh_freq='1 minute',
    feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('PURCHASES_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('PURCHASES_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('PURCHASES_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('PURCHASES_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '1w' ).alias('PURCHASES_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '6h' ).alias('PURCHASES_AMT_L6H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('PURCHASES_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '48h').alias('PURCHASES_AMT_L48H'),
        Feature.sum(   'AMOUNT_USD',   '1w' ).alias('PURCHASES_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_MAX_L1H'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('PURCHASES_MAX_L24H'),
        Feature.max(   'AMOUNT_USD',   '1w' ).alias('PURCHASES_MAX_L1WK'),
        Feature.min(   'AMOUNT_USD',   '24h').alias('PURCHASES_MIN_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '1h' ).alias('DISTINCT_MERCHANTS_L1H'),
        Feature.approx_count_distinct('MERCHANT_ID', '6h' ).alias('DISTINCT_MERCHANTS_L6H'),
        Feature.approx_count_distinct('MERCHANT_ID', '24h').alias('DISTINCT_MERCHANTS_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '1w' ).alias('DISTINCT_MERCHANTS_L1WK'),
        Feature.approx_count_distinct('WALLET_DPAN', '24h').alias('DISTINCT_DPANS_L24H'),
        Feature.approx_count_distinct('WALLET_DPAN', '1w' ).alias('DISTINCT_DPANS_L1WK'),
        Feature.approx_count_distinct('AMOUNT_USD',  '1h' ).alias('DISTINCT_AMOUNTS_L1H'),
        Feature.count('IS_GBR', '24h').alias('PURCHASES_GBR_NUM_L24H'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Customer velocity — CONTINUOUS, sub-2s freshness',
)
fs.register_feature_view(customer_fv, version='V1', block=True)
print('  Registered: FRAUD_CUSTOMER_VELOCITY')

In [ ]:
# ── Merchant velocity ────────────────────────────────────────────────────────
merchant_fv = FeatureView(
    name='FRAUD_MERCHANT_VELOCITY',
    entities=[merchant_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('MERCHANT_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('MERCHANT_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '1w' ).alias('MERCHANT_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '1w' ).alias('MERCHANT_TXN_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('MERCHANT_MAX_TXN_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('MERCHANT_UNIQUE_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('MERCHANT_UNIQUE_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1w' ).alias('MERCHANT_UNIQUE_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Merchant velocity — CONTINUOUS',
)
fs.register_feature_view(merchant_fv, version='V1', block=True)
print('  Registered: FRAUD_MERCHANT_VELOCITY')

# ── Wallet DPAN velocity ──────────────────────────────────────────────────────
dpan_fv = FeatureView(
    name='FRAUD_DPAN_VELOCITY',
    entities=[dpan_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('DPAN_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('DPAN_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('DPAN_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('DPAN_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '1w' ).alias('DPAN_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('DPAN_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '1w' ).alias('DPAN_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('DPAN_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1w' ).alias('DPAN_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='DPAN velocity — shared DPAN = fraud signal',
)
fs.register_feature_view(dpan_fv, version='V1', block=True)
print('  Registered: FRAUD_DPAN_VELOCITY')

# ── IP Address velocity ───────────────────────────────────────────────────────
ip_fv = FeatureView(
    name='FRAUD_IP_VELOCITY',
    entities=[ip_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('IP_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('IP_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('IP_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('IP_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '1w' ).alias('IP_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('IP_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '1w' ).alias('IP_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('IP_DISTINCT_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('IP_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1w' ).alias('IP_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='IP velocity — shared IP = bot farm signal',
)
fs.register_feature_view(ip_fv, version='V1', block=True)
print('  Registered: FRAUD_IP_VELOCITY')
print('\nAll 4 entity stream feature views registered.')

## 2.5 Customer Profile Feature View (Batch, Daily Refresh)

Static lifetime features that change slowly. Served from a DT-backed batch online feature view refreshed daily.

In [ ]:
profile_src = session.table('FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_fv = FeatureView(
    name='FRAUD_CUSTOMER_PROFILE',
    entities=[customer_entity],
    feature_df=profile_src,
    refresh_freq='1 day',
    online_config=OnlineConfig(enable=True, target_lag='1h', store_type=OnlineStoreType.POSTGRES),
    desc='Customer profile: lifetime stats, account age. Batch online FV, daily refresh.',
)
fs.register_feature_view(profile_fv, version='V1', block=True)
print('Registered: FRAUD_CUSTOMER_PROFILE (batch, daily refresh)')

## 2.6 Freshness Benchmark (RUN LIVE)

Measures time from REST ingest to feature visible at query time. Target: < 2 seconds.

In [ ]:
test_cust = f'CUST_{random.randint(100000, 199999)}'

b = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
    'name': 'FRAUD_CUSTOMER_VELOCITY', 'version': 'V1', 'object_type': 'feature_view',
    'request_rows': [{'entity': {'CUSTOMER_ID': test_cust}}], 'features': ['PURCHASES_NUM_L1H'],
}, timeout=10)
baseline_count = 0
if b.status_code == 200:
    rows = b.json().get('rows', [])
    if rows:
        baseline_count = rows[0].get('PURCHASES_NUM_L1H', 0) or 0
print(f'Baseline PURCHASES_NUM_L1H for {test_cust}: {baseline_count}')

ingest_ts = datetime.utcnow()
ir = requests.post(f'{INGEST_URL}/api/v1/ingest', headers=HEADERS, json={
    'dry_run': False,
    'records': {'FRAUD_TXN_EVENTS': [{
        'CUSTOMER_ID': test_cust, 'MERCHANT_ID': f'MERCH_{random.randint(1,5000)}',
        'WALLET_DPAN': f'DPAN_{random.randint(1,50000)}', 'IP_ADDRESS': f'IP_{random.randint(1,10000)}',
        'AMOUNT_USD': round(random.uniform(10, 500), 2), 'IS_GBR': 1.0,
        'EVENT_TS': ingest_ts.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }]},
}, timeout=10)
print(f'Ingest: HTTP {ir.status_code}')
start = time.time()

print('\nPolling...')
updated = False
for i in range(120):
    time.sleep(0.25)
    r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
        'name': 'FRAUD_CUSTOMER_VELOCITY', 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {'CUSTOMER_ID': test_cust}}], 'features': ['PURCHASES_NUM_L1H'],
    }, timeout=10)
    if r.status_code == 200:
        rows = r.json().get('rows', [])
        if rows and (rows[0].get('PURCHASES_NUM_L1H', 0) or 0) > baseline_count:
            ms = (time.time() - start) * 1000
            print(f'\nFeature updated in {ms:.0f}ms')
            print(f'PURCHASES_NUM_L1H: {baseline_count} → {rows[0]["PURCHASES_NUM_L1H"]}')
            updated = True
            break
    if i % 8 == 0:
        print(f'  [{i*0.25:.1f}s] waiting...')
if not updated:
    print('Not updated within 30s')

## 2.7 Latency Benchmark

10 warmup + 200 measured requests. Single-entity and 4-entity concurrent (per-transaction) lookups using real entity keys.

In [ ]:
N_WARMUP, N_REQUESTS = 10, 200
samples = session.sql('SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (250 ROWS)').collect()
print(f'Sampled {len(samples)} entity keys\n')

def qlat(name, key_col, key_val):
    t0 = time.perf_counter()
    r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
        'name': name, 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {key_col: key_val}}],
    }, timeout=10)
    r.raise_for_status()
    return (time.perf_counter() - t0) * 1000

# Benchmark 1: single entity
print('='*55); print('BENCHMARK 1: Single-entity (customer)'); print('='*55)
lat_s = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    lat = qlat('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID'])
    if i >= N_WARMUP: lat_s.append(lat)
    if i == N_WARMUP - 1: print('  Warmup complete')
arr_s = np.array(lat_s)
print(f'  p50: {np.percentile(arr_s,50):.1f}ms  p95: {np.percentile(arr_s,95):.1f}ms  p99: {np.percentile(arr_s,99):.1f}ms')

# Benchmark 2: 4-entity concurrent
print(f'\n{"="*55}'); print('BENCHMARK 2: 4-entity concurrent (per-transaction)'); print('='*55)
def q_all(row):
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        fs_ = [ex.submit(qlat, 'FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID']),
               ex.submit(qlat, 'FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID', row['MERCHANT_ID']),
               ex.submit(qlat, 'FRAUD_DPAN_VELOCITY',     'WALLET_DPAN',  row['WALLET_DPAN']),
               ex.submit(qlat, 'FRAUD_IP_VELOCITY',       'IP_ADDRESS',   row['IP_ADDRESS'])]
    return max(f.result() for f in fs_)

lat_m = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    wall = q_all(row)
    if i >= N_WARMUP: lat_m.append(wall)
    if i == N_WARMUP - 1: print('  Warmup complete')
arr_m = np.array(lat_m)
print(f'  p50: {np.percentile(arr_m,50):.1f}ms  p95: {np.percentile(arr_m,95):.1f}ms  p99: {np.percentile(arr_m,99):.1f}ms')
print(f'  wall time = slowest of 4 parallel lookups')

---
## Summary

| Component | Freshness | Method |
|---|---|---|
| FRAUD_CUSTOMER_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_MERCHANT_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_DPAN_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_IP_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_CUSTOMER_PROFILE | Daily | Batch online FV |

**Platform cost:** ~$200-500/month (Online Service). No 24/7 DT warehouse.

**Next:** NB03 — train XGBoost from the transactions table.